In [75]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv

In [76]:
load_dotenv()

CLIENT_ID = os.getenv('CLIENT_ID') or os.getenv('SPOTIPY_CLIENT_ID')
CLIENT_SECRET = os.getenv('CLIENT_SECRET') or os.getenv('SPOTIPY_CLIENT_SECRET')
REDIRECT_URI = os.getenv('REDIRECT_URI') or os.getenv('SPOTIPY_REDIRECT_URI')
LASTFM_API_KEY = os.getenv('LASTFM_API_KEY')
playlist_id = "7vNFsusz16Ao85IR9SU5cd"

if not CLIENT_ID or not CLIENT_SECRET or not REDIRECT_URI:
    raise ValueError(
        "Spotify credentials are missing. Set CLIENT_ID, CLIENT_SECRET, REDIRECT_URI "
        "or SPOTIPY_CLIENT_ID, SPOTIPY_CLIENT_SECRET, SPOTIPY_REDIRECT_URI in your environment or .env file."
    )

In [77]:
# --- 1. AUTENTICACIÓN ---
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri=REDIRECT_URI,
    scope="playlist-read-private playlist-read-collaborative"
))


try:
    current_user = sp.current_user()
    print("Authenticated as:", current_user.get("display_name", current_user.get("id")))
except Exception as auth_error:
    print("Authentication failed:", auth_error)
    raise


# --- 2. OBTENER ITEMS DE LA PLAYLIST ---
try:
    playlist_items = sp.playlist_items(
        playlist_id,
        limit=100,
        market=None,
        additional_types=("track",)
    )
    print("Fetched playlist items:", len(playlist_items.get("items", [])))
except Exception as playlist_error:
    print("Playlist items fetch failed:", playlist_error)
    raise

# --- 3. EXTRACCIÓN SEGURA DE DATOS (A PRUEBA DE ERROR 403) ---
track_data = []

for item in playlist_items.get("items", []):
    track = item.get("track") or item.get("item")
    
    # Validar que exista el track y tenga ID
    if not isinstance(track, dict) or not track.get("id"):
        continue
        
    track_name = track.get("name", "Nombre desconocido")
    artists = track.get("artists", [])
    artist_name = ", ".join([artist.get("name", "") for artist in artists]) if artists else "Artista desconocido"
    
    # Tomamos la popularidad que Spotify esté dispuesto a darnos aquí (si es 0, es por su bloqueo)
    popularity = track.get("popularity")

    track_data.append({
        "track_name": track_name,
        "artist_name": artist_name,
        "popularity": popularity,
        "key": "Requiere API Externa" 
    })

if not track_data:
    raise ValueError("No se encontraron canciones válidas.")

# --- 4. CREACIÓN DEL DATAFRAME ---
playlist_df = pd.DataFrame(track_data)
playlist_df = playlist_df[["track_name", "artist_name", "popularity", "key"]]

print("\n--- RESULTADO FINAL ---")
print(playlist_df.head(10))

Authenticated as: Danseur º
Fetched playlist items: 100

--- RESULTADO FINAL ---
                                      track_name  \
0                                    stupid song   
1                                    Billie Jean   
2                                           SWIM   
3                              Beauty And A Beat   
4                                    Janice STFU   
5                               COPING MECHANISM   
6                                       Babydoll   
7                             National Treasures   
8  Ran To Atlanta (feat. Future & Molly Santana)   
9                                Whisper My Name   

                    artist_name popularity                   key  
0                Olivia Rodrigo       None  Requiere API Externa  
1               Michael Jackson       None  Requiere API Externa  
2                           BTS       None  Requiere API Externa  
3    Justin Bieber, Nicki Minaj       None  Requiere API Externa  
4          

In [78]:
playlist_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   track_name   100 non-null    str   
 1   artist_name  100 non-null    str   
 2   popularity   0 non-null      object
 3   key          100 non-null    str   
dtypes: object(1), str(3)
memory usage: 3.3+ KB


In [79]:
playlist_df.head()

,track_name,artist_name,popularity,key
0,stupid song,Olivia Rodrigo,None,Requiere API Externa
1,Billie Jean,Michael Jackson,None,Requiere API Externa
2,SWIM,BTS,None,Requiere API Externa
3,Beauty And A Beat,"Justin Bieber, Nicki Minaj",None,Requiere API Externa
4,Janice STFU,Drake,None,Requiere API Externa


In [80]:
import requests
import pandas as pd
import time
import re  # Importamos la librería de expresiones regulares para limpiar texto

# Tu API Key
LASTFM_URL = "http://ws.audioscrobbler.com/2.0/"

# --- FUNCIÓN LIMPIADORA ---
def limpiar_texto(texto):
    # Elimina todo lo que esté entre paréntesis (ej. "(feat. Drake)")
    texto = re.sub(r'\(.*?\)', '', texto)
    # Elimina todo lo que esté entre corchetes (ej. "[Radio Edit]")
    texto = re.sub(r'\[.*?\]', '', texto)
    # Elimina todo lo que esté después de un guion (ej. "- Remastered")
    texto = texto.split(' - ')[0]
    return texto.strip()

print("\n--- INICIANDO EXTRACCIÓN LIMPIA DE LAST.FM ---")

for track in track_data:
    # 1. Limpiamos el nombre de la canción
    track_name_clean = limpiar_texto(track["track_name"])
    
    # 2. Limpiamos el artista (Spotify a veces separa con '&' además de comas)
    primary_artist = track["artist_name"].split(",")[0].split(" & ")[0].strip()
    
    payload = {
        "method": "track.getInfo",
        "api_key": LASTFM_API_KEY,
        "artist": primary_artist,
        "track": track_name_clean,
        "format": "json",
        "autocorrect": 1
    }
    
    try:
        response = requests.get(LASTFM_URL, params=payload)
        data = response.json()
        
        if "track" in data and "playcount" in data["track"]:
            playcount = int(data["track"]["playcount"])
            track["popularity"] = playcount
            print(f"✅ Éxito: {track_name_clean} -> {playcount:,} repros")
        else:
            print(f"❌ No encontrada: {track_name_clean}")
            track["popularity"] = 0
            
    except Exception as e:
        print(f"Error de conexión con {track_name_clean}: {e}")
        track["popularity"] = 0
        
    time.sleep(0.2)

# --- CREACIÓN DEL DATAFRAME FINAL ---
playlist_df = pd.DataFrame(track_data)
playlist_df = playlist_df.sort_values(by="popularity", ascending=False)

print("\n--- DATAFRAME ACTUALIZADO ---")
print(playlist_df.head(15))


--- INICIANDO EXTRACCIÓN LIMPIA DE LAST.FM ---
✅ Éxito: stupid song -> 3,757,975 repros
✅ Éxito: Billie Jean -> 28,403,370 repros
✅ Éxito: SWIM -> 257,045,465 repros
✅ Éxito: Beauty And A Beat -> 18,550,859 repros
✅ Éxito: Janice STFU -> 2,984,096 repros
✅ Éxito: COPING MECHANISM -> 28,568 repros
✅ Éxito: Babydoll -> 27,458,874 repros
✅ Éxito: National Treasures -> 1,948,833 repros
✅ Éxito: Ran To Atlanta -> 59,549 repros
✅ Éxito: Whisper My Name -> 1,866,433 repros
✅ Éxito: drop dead -> 12,811,325 repros
✅ Éxito: DAISIES -> 7,857,409 repros
✅ Éxito: System -> 743 repros
✅ Éxito: DtMF -> 18,037,755 repros
✅ Éxito: Man I Need -> 10,849,711 repros
✅ Éxito: Earrings -> 12,773,217 repros
✅ Éxito: The One That Got Away -> 21,889,160 repros
✅ Éxito: back to friends -> 26,368,312 repros
✅ Éxito: BAILE INoLVIDABLE -> 14,466,530 repros
✅ Éxito: The Fate of Ophelia -> 26,604,678 repros
✅ Éxito: Golden -> 17,858,316 repros
✅ Éxito: Money or Life -> 80 repros
✅ Éxito: Area Code -> 555 repros
✅ Éx

In [83]:
playlist_df.info()

<class 'pandas.DataFrame'>
Index: 100 entries, 2 to 21
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   track_name   100 non-null    str  
 1   artist_name  100 non-null    str  
 2   popularity   100 non-null    int64
 3   key          100 non-null    str  
dtypes: int64(1), str(3)
memory usage: 3.9 KB


In [84]:
playlist_df['popularity'].unique()

array([257045465,  56615029,  56537818,  37036003,  36009780,  31950923,
        30586760,  28403370,  28375865,  27503416,  27458874,  27177182,
        27048564,  26604678,  26368312,  26340161,  23963452,  22175210,
        21889160,  18824262,  18550859,  18037755,  17858316,  15674168,
        15284090,  14466530,  14010461,  13468885,  12811325,  12773217,
        12372241,  11491365,  11142482,  11073316,  10849711,  10276648,
        10233579,  10042652,   8327865,   7869676,   7857409,   7418487,
         7160061,   6723532,   6151919,   5815370,   5230768,   5024277,
         5014264,   4955703,   4387559,   3857000,   3757975,   3578872,
         3234276,   2984096,   2841690,   2471239,   2353769,   2351935,
         2283696,   1948833,   1886496,   1866433,   1756927,   1647673,
         1539782,   1329792,   1329269,   1217815,   1198748,   1034598,
         1011007,    721411,    523780,    519250,    342255,    313781,
          232754,    129541,    116544,    115251, 